# LoRA demo: Hugging Face PEFT + tiny GPT-2

Transfer Learning, кастомізація LLM і RAG

У цьому notebook ми:

1. завантажимо маленьку causal language model;
2. подивимося, скільки параметрів має base model;
3. додамо LoRA adapters через `peft`;
4. навчимо тільки adapter-параметри на маленькому synthetic dataset;
5. порівняємо генерацію до і після LoRA;
6. збережемо adapter окремо від base model.

За замовчуванням використовується `distilbert/distilgpt2`, бо він достатньо малий для локального demo, але помітно стабільніший за `sshleifer/tiny-gpt2`. Якщо потрібен максимально швидкий запуск на слабкому CPU, можна змінити `MODEL_NAME` на `sshleifer/tiny-gpt2`.


## Встановлення бібліотек


In [ ]:
# !pip -q install "torch>=2.2" "transformers>=4.40" "peft>=0.18" "accelerate>=0.28"


## 1. Імпорти та налаштування

`RUN_TRAINING = True` запускає коротке навчання adapter. Якщо потрібно тільки показати структуру LoRA без тренування, змініть на `False`.


In [ ]:
from pathlib import Path
import random

import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, get_linear_schedule_with_warmup
from peft import LoraConfig, PeftModel, TaskType, get_peft_model

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

# Better classroom default. For the fastest CPU-only demo use: "sshleifer/tiny-gpt2".
MODEL_NAME = "distilbert/distilgpt2"

OUTPUT_DIR = Path("models/lora_gpt2_support_adapter")
RUN_TRAINING = True
NUM_EPOCHS = 8
BATCH_SIZE = 2
LEARNING_RATE = 0.0002
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
MAX_GRAD_NORM = 1.0
TRAINING_REPEAT_FACTOR = 4
MAX_LENGTH = 384

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print("Device:", DEVICE)


## 2. Завантажуємо tokenizer і base model

GPT-2-подібні моделі не мають окремого `pad_token`, тому для batch training зазвичай ставлять `pad_token = eos_token`.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
base_model.config.pad_token_id = tokenizer.pad_token_id
base_model.to(DEVICE)

print("Model:", MODEL_NAME)
print("Pad token:", repr(tokenizer.pad_token), "id:", tokenizer.pad_token_id)


## 3. Допоміжні функції

Порахуємо параметри і зробимо функцію для короткої генерації. Це допоможе порівняти base model і LoRA model.


In [ ]:
def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


def print_parameter_report(model, title):
    total, trainable = count_parameters(model)
    pct = 100 * trainable / total
    print(title)
    print(f"Total parameters:     {total:,}")
    print(f"Trainable parameters: {trainable:,} ({pct:.4f}%)")


def build_prompt(user_text):
    return (
        "### Запит клієнта:\n"
        f"{user_text}\n\n"
        "### Відповідь асистента:\n"
    )


def generate_text(model, user_text, max_new_tokens=70):
    model.eval()
    prompt = build_prompt(user_text)
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.15,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)


print_parameter_report(base_model, "Base model before LoRA")


## 4. Перевіряємо baseline-відповідь

До fine-tuning модель не знає наш формат відповіді. На tiny model текст може бути майже випадковим.


In [ ]:
test_request = "Клієнт пише, що оплатив тариф, але підписка не активувалася."
print(generate_text(base_model, test_request))


## 5. Знаходимо GPT-2 modules для LoRA

У GPT-2 основні projection layers мають назви на кшталт `c_attn` і `c_proj`. Саме до них часто додають LoRA adapters.


In [ ]:
for name, module in base_model.named_modules():
    if any(part in name for part in ["c_attn", "c_proj"]):
        print(f"{name:<45} {module.__class__.__name__}")


## 6. Додаємо LoRA adapters через PEFT

LoRA заморожує base model і додає маленькі trainable матриці до вибраних шарів. Тому ми навчаємо лише невелику частку параметрів.

Ключові параметри:

- `r` - rank adapter-матриць;
- `lora_alpha` - масштаб LoRA-оновлення;
- `target_modules` - куди саме вставити adapters;
- `task_type="CAUSAL_LM"` - тип задачі для autoregressive language model.


In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=["c_attn", "c_proj"],
    fan_in_fan_out=True,
)

lora_model = get_peft_model(base_model, lora_config)
lora_model.to(DEVICE)

print_parameter_report(lora_model, "Model after adding LoRA")
lora_model.print_trainable_parameters()


## 7. Малий instruction dataset

Задача: навчити модель відповідати у простому форматі для support-router:

- визначити категорію звернення;
- дати коротку відповідь клієнту;
- запропонувати наступну дію.


In [ ]:
training_examples = [
    {
        "user": "The customer cannot log in because the SMS code does not arrive.",
        "assistant": "Category: TECHNICAL\nResponse: Please check the phone number and request the code again in a few minutes. If the code still does not arrive, we will forward the issue to technical support.\nNext action: check SMS delivery.",
    },
    {
        "user": "After the update, the app freezes on the login screen.",
        "assistant": "Category: TECHNICAL\nResponse: Try clearing the app cache or reinstalling the app. If the issue remains, we need the device model and OS version.\nNext action: collect technical details.",
    },
    {
        "user": "The user sees a 500 error when opening the report.",
        "assistant": "Category: TECHNICAL\nResponse: This looks like a server-side error. We will check the event logs and let you know when the report is available again.\nNext action: create a technical ticket.",
    },
    {
        "user": "The card was charged twice for one subscription plan.",
        "assistant": "Category: BILLING\nResponse: We will check the duplicate charge and refund the extra payment if the transaction is confirmed.\nNext action: request the date, amount, and last 4 digits of the card.",
    },
    {
        "user": "The payment went through, but the subscription was not activated.",
        "assistant": "Category: BILLING\nResponse: Thank you for the message. We will check the payment status and manually activate the subscription if the payment was successful.\nNext action: check the payment id or receipt.",
    },
    {
        "user": "The customer asks for an invoice for accounting for the previous month.",
        "assistant": "Category: BILLING\nResponse: We can prepare an invoice for the requested period and send it to the account email.\nNext action: confirm the legal billing details.",
    },
    {
        "user": "The order was not delivered on the scheduled day.",
        "assistant": "Category: DELIVERY\nResponse: We will check the delivery status with the carrier and share the new expected arrival time.\nNext action: confirm the order number.",
    },
    {
        "user": "The customer wants to change the delivery address after placing the order.",
        "assistant": "Category: DELIVERY\nResponse: If the order has not yet been handed over to the carrier, we will change the delivery address.\nNext action: check the shipment status.",
    },
    {
        "user": "The package status is delivered, but the customer did not receive it.",
        "assistant": "Category: DELIVERY\nResponse: We will check the delivery confirmation and contact the carrier for more details.\nNext action: open a delivery investigation.",
    },
]

print("Examples:", len(training_examples))
print(training_examples[0]["user"])
print(training_examples[0]["assistant"])


## 8. Dataset для causal language modeling

Ми навчаємо модель продовжувати prompt правильною відповіддю, але loss рахуємо тільки на токенах відповіді assistant. Prompt-токени і padding отримують label `-100`, тому PyTorch ігнорує їх у `CrossEntropyLoss`.

Для цього dataset важливо не обрізати відповіді. Англійська версія прикладів коротша за українську, але `MAX_LENGTH = 384` залишає запас, щоб повні відповіді потрапляли в навчання.


In [ ]:
class SupportDataset(Dataset):
    def __init__(self, examples, tokenizer, max_length):
        self.rows = []
        self.truncated_rows = 0

        for item in examples:
            prompt = build_prompt(item["user"])
            full_text = prompt + item["assistant"] + tokenizer.eos_token

            full_length = len(tokenizer(full_text, add_special_tokens=False)["input_ids"])
            if full_length > max_length:
                self.truncated_rows += 1

            encoded = tokenizer(
                full_text,
                max_length=max_length,
                truncation=True,
                padding="max_length",
                return_tensors="pt",
            )
            prompt_encoded = tokenizer(
                prompt,
                max_length=max_length,
                truncation=True,
                add_special_tokens=False,
                return_tensors="pt",
            )

            input_ids = encoded["input_ids"].squeeze(0)
            attention_mask = encoded["attention_mask"].squeeze(0)
            prompt_length = prompt_encoded["input_ids"].shape[1]

            labels = input_ids.clone()
            labels[attention_mask == 0] = -100
            labels[:prompt_length] = -100

            if (labels != -100).sum().item() == 0:
                raise ValueError("MAX_LENGTH is too small: no assistant tokens left for training.")

            self.rows.append({
                "input_ids": input_ids,
                "attention_mask": attention_mask,
                "labels": labels,
            })

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        return self.rows[idx]


train_examples = training_examples * TRAINING_REPEAT_FACTOR
dataset = SupportDataset(train_examples, tokenizer, MAX_LENGTH)
generator = torch.Generator().manual_seed(SEED)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, generator=generator)

batch = next(iter(dataloader))
print({key: value.shape for key, value in batch.items()})
print("Training rows:", len(dataset), "| batches:", len(dataloader))
print("Truncated rows:", dataset.truncated_rows)


## 9. Коротке тренування LoRA

Навчаються тільки adapter-параметри. Оскільки dataset дуже малий, ми повторюємо приклади кілька разів в межах навчального набору, щоб кожна epoch мала більше optimizer steps і loss був менш шумним.

Для стабільності використовуємо менший learning rate, warmup + linear decay, weight decay і gradient clipping.


In [ ]:
if RUN_TRAINING:
    trainable_params = [p for p in lora_model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(
        trainable_params,
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    num_training_steps = NUM_EPOCHS * len(dataloader)
    num_warmup_steps = max(1, int(num_training_steps * WARMUP_RATIO))
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=num_training_steps,
    )

    print("Training steps:", num_training_steps, "| warmup steps:", num_warmup_steps)

    step = 0
    for epoch in range(NUM_EPOCHS):
        lora_model.train()
        epoch_loss = 0.0

        for batch in dataloader:
            batch = {key: value.to(DEVICE) for key, value in batch.items()}
            optimizer.zero_grad(set_to_none=True)

            outputs = lora_model(**batch)
            loss = outputs.loss
            loss.backward()

            torch.nn.utils.clip_grad_norm_(trainable_params, MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()

            epoch_loss += loss.item()
            step += 1

        mean_loss = epoch_loss / len(dataloader)
        print(f"Epoch {epoch + 1:02d}/{NUM_EPOCHS} | loss: {mean_loss:.4f}")
else:
    print("RUN_TRAINING = False, training skipped.")


## 10. Генерація після LoRA

На такому маленькому dataset результат не має бути ідеальним. Для демонстрації важливо, що модель починає наслідувати формат: `Category`, `Response`, `Next action`.


In [ ]:
for request in [
    "Клієнт пише, що оплатив тариф, але підписка не активувалася.",
    "Клієнт не отримав посилку, хоча статус показує delivered.",
    "Після входу в кабінет сторінка звітів відкривається з помилкою 500.",
]:
    print("=" * 90)
    print(generate_text(lora_model, request))


## 11. Зберігаємо LoRA adapter

PEFT зберігає adapter окремо від base model. Це зручно: base model може залишатися спільною, а adapter-файли маленькі і відповідають конкретній задачі або стилю.


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
lora_model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Saved adapter to:", OUTPUT_DIR.resolve())
print("Files:")
for file in sorted(OUTPUT_DIR.iterdir()):
    print("-", file.name)

## 12. Завантажуємо adapter назад

Для inference потрібно завантажити ту саму base model і накласти на неї adapter через `PeftModel.from_pretrained`.


In [ ]:
base_for_inference = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
base_for_inference.config.pad_token_id = tokenizer.pad_token_id

loaded_lora_model = PeftModel.from_pretrained(base_for_inference, OUTPUT_DIR)
loaded_lora_model.to(DEVICE)

print(generate_text(loaded_lora_model, "Клієнт просить рахунок для бухгалтерії за минулий місяць."))


## 13. Optional: merge adapter into base model

Для deployment іноді adapter зливають із base model. Це робить одну звичайну model checkpoint, але втрачає гнучкість швидко перемикати adapters.


In [ ]:
# merged_model = loaded_lora_model.merge_and_unload()
# merged_model.save_pretrained("models/lora_gpt2_support_merged")
# tokenizer.save_pretrained("models/lora_gpt2_support_merged")


## Довідка

Офіційні матеріали, які корисно відкрити після demo:

- [Hugging Face PEFT: LoRA package reference](https://huggingface.co/docs/peft/package_reference/lora)
- [Transformers: Parameter-efficient fine-tuning](https://huggingface.co/docs/transformers/peft)
- [PEFT model API](https://huggingface.co/docs/peft/package_reference/peft_model)


## Ключова ідея

Full fine-tuning змінює всі ваги моделі. LoRA залишає base model замороженою і навчає маленькі adapter-матриці у вибраних шарах. Тому adapter дешевше тренувати, зберігати і переносити між середовищами.
